# 3 · Evaluation: did this build keep its promises?

Notebook 2 explained one turn. This notebook measures many: it replays fixed
conversations, checks what the agent actually did, and compares runs, so that
you know whether a change made the assistant better or worse.

| | |
|---|---|
| **Time** | about 30 minutes, most of it waiting for replays |
| **You need** | the stack from [1 · Getting Started](1_Getting_Started.ipynb) |
| **Reference** | [tests/evaluation/README.md](../tests/evaluation/README.md), [datasets/val/README.md](../tests/evaluation/datasets/val/README.md) |

**You will learn to:**

1. Read a scenario: fixed shopper words, plus checks on state.
2. Run a replay and read its report and transcripts.
3. Trace a failure back to its turn and its trace.
4. Use repeats to tell a flaky behaviour from a broken one.
5. Compare two runs, scenario by scenario.
6. Write a scenario of your own.
7. Know when to use the Challenger and the Judge instead.

**How to use this notebook.** Run the cells in order. Each step says what to
**Run** or **Do**, then shows what **You should see**. Replies vary between runs.
Pass or fail should not, and when it does, that is what step 6 is for.

## 1. What we measure, and why

**Check state, not wording.** A reply can read perfectly and still be wrong:
it can confirm a size nobody asked for, or say a dress is in the cart when it
is not. So each check reads what the services hold: the cart after the turn,
the products streamed, the tools called. "It should ask rather than act" is
checked as *the cart did not change*, not by looking for a question mark.

There are two ways to evaluate, and each answers a different question:

| | Replay | Challenger |
|---|---|---|
| **The shopper's words** | frozen in a script | generated fresh by a model each run |
| **Checked by** | assertions on cart, products and tools | a Judge model reading the conversation |
| **Answers** | did this build keep its promises? | what is broken that nobody thought of? |
| **Run** | before every merge | nightly, and after a large change |

Neither replaces the other. When the Challenger finds a bug, freeze those turns
as a replay scenario, and the bug cannot come back unnoticed. Most of this
notebook is about replay.

## 2. Before you start

Replay checks tool calls, which the chain-server only returns when
`EXPOSE_AGENT_DIAGNOSTICS=true`. Notebook 1 turns it on.

**Run** the next cell. It installs the two libraries the replay uses into this
kernel, and checks the setting inside the running container. Ignore any note
about restarting the kernel.

In [ ]:
%pip install --quiet requests pyyaml

import json, subprocess, sys
from collections import defaultdict
from pathlib import Path
from IPython.display import Markdown, display
from helpers import *

EVAL = REPO / "tests/evaluation"
RESULTS = EVAL / "results/val"
env = subprocess.run(["docker", "exec", "chain-server", "printenv", "EXPOSE_AGENT_DIAGNOSTICS"],
                     capture_output=True, text=True).stdout.strip()
print("chain-server:", status(f"{CHAIN_SERVER}/health"), "| EXPOSE_AGENT_DIAGNOSTICS =", env or "(not set)")

**You should see:**

```
chain-server: 200 | EXPOSE_AGENT_DIAGNOSTICS = true
```

**If not:** add `export EXPOSE_AGENT_DIAGNOSTICS=true` to `.env`, then run
`source .env && docker compose up -d chain-server` from the repo root. Keep it
off in production: diagnostics contain tool arguments and internal product ids.

## 3. Anatomy of a scenario

Scenarios are YAML files in `tests/evaluation/datasets/val/scripts/`:

| Folder | Holds |
|---|---|
| `journeys/` (`J01`-`J20`) | long conversations, 6 to 20 turns, where context builds up |
| `probes/` (`P01`-`P30`) | 1 to 4 turns, each checking a single behaviour |
| `regression/` (`R..`) | bugs that were fixed, kept fixed |

**Run** the next cell to read one probe:

In [ ]:
probe = EVAL / "datasets/val/scripts/probes/P08_cart_two_sizes_are_two_lines.yaml"
print(probe.read_text())

**You should see** the scenario, with each turn's words and checks:

```yaml
id: P08_cart_two_sizes_are_two_lines
covers: [cart-management, sizes]
why: >
  A size is a line, not a quantity, and a size nobody named is not their size. ...
turns:
  - say: show me black dresses in a size 2
    expect:
      products_min: 1
      every_product: {primary_color: black}

  - say: add the Black Satin Lace-Up Dress to my cart
    expect:
      cart_unchanged: true            # it must ask the size, not choose one
  ...
```

Every scenario says **why** it exists. The next person to see it fail needs to
know whether it guards a real rule.

| Check | Answered by |
|---|---|
| `cart:` | the cart service: name, size, quantity, and the number of lines |
| `cart_unchanged: true` | the cart before and after the turn |
| `products_min`, `products_max` | the products the turn streamed |
| `every_product: {field: value}` | the catalog, for each product shown |
| `no_product_named: [...]` | the products the turn streamed |
| `tools_used`, `tools_not_used` | the turn's diagnostics |

## 4. Run a replay

Two probes: `P07` (no size given, so no add) and `P08` (a size used in a search
is not a size the shopper chose). P08 is a **known failure** in this release,
which makes it a good one to learn to read.

**Run** the next cell. It takes two to three minutes, one conversation at a time.

In [ ]:
FIRST = "notebook-first"
!cd .. && {sys.executable} -m tests.evaluation.src.replay --only P07,P08 --label {FIRST} 2>&1 | tail -8

**You should see** one line per scenario, each failed check under its
scenario, a tally, and where the results went:

```
  2 runs, one at a time, label notebook-first
  ok   P07_cart_size_required
  FAIL P08_cart_two_sizes_are_two_lines
       turn 2 cart_unchanged: cart went from [] to [('Black Satin Lace-Up Dress', '2', 1)]

  pass 1  fail 1  error 0
  -> .../tests/evaluation/results/val/notebook-first/report.md
  -> .../tests/evaluation/results/val/notebook-first/transcripts/  (2 conversations to read)
```

`error` counts runs that could not finish, for example because a service was
down. It is not a verdict on the agent.

Runs go one at a time by default, so timings are comparable. `--parallel` runs
up to six at once. Past that nothing finishes sooner, because the model
endpoint is the bottleneck.

## 5. Read the report

Each run writes three things to `tests/evaluation/results/val/<label>/`, which
git ignores:

| File | Holds |
|---|---|
| `report.md` | one row per scenario, with failures listed underneath |
| `transcripts/<id>-<n>.md` | the conversation, with the cart after every turn |
| `raw/<id>-<n>.json` | everything, for a script to read |

**Run:**

In [ ]:
display(Markdown((RESULTS / FIRST / "report.md").read_text()))

**You should see** a table and, under it, each failed check with what was found:

| scenario | outcome | checks | covers |
|---|---|---|---|
| `P07_cart_size_required` | pass | 5 | cart-management, sizes |
| `P08_cart_two_sizes_are_two_lines` | fail | 5 | cart-management, sizes |

```
### P08_cart_two_sizes_are_two_lines
- turn 2 cart_unchanged: cart went from [] to [('Black Satin Lace-Up Dress', '2', 1)]
```

Read that line as: on turn 2 the cart should have stayed empty, and instead it
gained the dress in size 2.

## 6. Trace a failure to its cause

A failed check tells you **which turn** broke. The transcript shows that turn:
the words, the cart, and what the tools were asked. It is the artifact to
judge from, by eye or by handing it to a model.

**Run** the next cell. It prints the failing turn of P08's transcript.

In [ ]:
def turn_of(transcript: Path, number: int) -> str:
    sections = transcript.read_text().split("\n## ")
    return "## " + next(s for s in sections if s.startswith(f"{number}. "))

run = json.loads((RESULTS / FIRST / "raw/P08_cart_two_sizes_are_two_lines-0.json").read_text())
print("outcome:", run["outcome"], "| failed:", run["failed_checks"])
print("conversation:", run["identity"]["conversation_id"], "\n")
print(turn_of(RESULTS / FIRST / "transcripts/P08_cart_two_sizes_are_two_lines-0.md", 2)[:1500])

**You should see** the failure, the conversation id, then turn 2 with the cart
printed under the reply:

```
outcome: fail | failed: ["turn 2 cart_unchanged: cart went from [] to [('Black Satin Lace-Up Dress', '2', 1)]"]
conversation: notebook-first-P08_cart_two_sizes_are_two_lines-0

## 2. add the Black Satin Lace-Up Dress to my cart

Added the **Black Satin Lace-Up Dress** in size 2 to your cart. ...

> **Cart: 1 x Black Satin Lace-Up Dress (size 2)**
> 3.6s · 0 products · tools ['activate_shopper_skills_tool', 'add_cart_items_tool']
> **FAILED** `cart_unchanged` — cart went from [] to [('Black Satin Lace-Up Dress', '2', 1)]

<details><summary>what the tools were asked — 2 calls</summary>
...
2. add_cart_items_tool
{"items": [{"product_ref": "generated:3185c59c1cab8b83", "expected_display_name": "Black Satin Lace-Up Dress", "size": "2"}]}
```

The transcript also lists what the tools were asked and what memory the turn
read. Here the add call carries `"size": "2"`, although the shopper said no size
on this turn.

The shopper used size 2 only to narrow the search on turn 1. The agent carried
it into a purchase on turn 2 without asking.

**Do**, to see why: open [2 · Observability](2_Observability.ipynb), set
`SESSION` to the conversation id printed above, and follow steps 4 to 7. The
add call's arguments show where the size came from. Q2 shows whether the rule
"a search size is not a purchase size" was in front of the model.

## 7. Repeat: flaky or broken?

The model is not deterministic, so one run proves little. Run the same scenario
several times and count the passes. The strict measure is **pass^k**: the share
of scenarios that passed **every one** of k runs. A shopper meets the agent
many times, so "usually right" is not good enough for a cart.

**Run** the next cell. It replays P07 three times, in about three minutes.

In [ ]:
REPEAT = "notebook-repeat"
!cd .. && {sys.executable} -m tests.evaluation.src.replay --only P07 --repeat 3 --label {REPEAT} 2>&1 | tail -5

In [ ]:
def outcomes(label):
    runs = defaultdict(list)
    for path in sorted((RESULTS / label / "raw").glob("*.json")):
        run = json.loads(path.read_text())
        runs[run["id"]].append(run["outcome"])
    return runs

for scenario, results in outcomes(REPEAT).items():
    passed = results.count("pass")
    print(f"{scenario:34} {passed}/{len(results)} passed  pass^{len(results)} = {passed == len(results)}")

**You should see** the replay's tally, `pass 3  fail 0  error 0`, then:

```
P07_cart_size_required             3/3 passed  pass^3 = True
```

How to read a tally:

| Result | Means | Do |
|---|---|---|
| 3/3 | holds | nothing |
| 1/3 or 2/3 | flaky: the rule sometimes reaches the model or sometimes wins | find a failing run's turn, then read its trace as in step 6 |
| 0/3 | broken | the rule is missing, or something overrides it |

Before a merge, run the scenarios a change touches with `--repeat 3` or more.

## 8. Compare two runs

To judge a change, run the same scenarios before and after it, then compare
them scenario by scenario. The overall pass count hides a fix and a regression
that cancel out.

**Run** the next cell. It compares the two runs you just made.

In [ ]:
def compare(label_a, label_b):
    a, b = outcomes(label_a), outcomes(label_b)
    for scenario in sorted(set(a) | set(b)):
        ra, rb = a.get(scenario, []), b.get(scenario, [])
        pa = f"{ra.count('pass')}/{len(ra)}" if ra else "-"
        pb = f"{rb.count('pass')}/{len(rb)}" if rb else "-"
        flag = ""
        if ra and rb:
            before, after = ra.count("pass") / len(ra), rb.count("pass") / len(rb)
            flag = "better" if after > before else "WORSE" if after < before else ""
        print(f"{scenario:34} {pa:>5} -> {pb:<5} {flag}")

compare(FIRST, REPEAT)

**You should see** each scenario's pass count in both runs. A scenario run in
only one of them shows `-`:

```
P07_cart_size_required               1/1 -> 3/3
P08_cart_two_sizes_are_two_lines     0/1 -> -
```

For a real comparison, use the same `--only` list and `--repeat` for both labels,
for example the full set before and after a change:

```bash
git switch staging     && python -m tests.evaluation.src.replay --label before --repeat 2
git switch my-change   && python -m tests.evaluation.src.replay --label after  --repeat 2
```

Rebuild the chain-server after each switch (`docker compose up -d --build chain-server`),
or both runs test the same build. Each transcript records the build it ran
against, so check its `Build:` line if in doubt. The full set takes about two
hours one at a time, or about forty minutes with `--parallel`.

## 9. Write a scenario of your own

A good scenario checks **one** promise, through state. Two rules from the
dataset's README:

- **Check state, never wording.** Assert the cart, not the reply.
- **Do not assert that a tool went uncalled when a refusal is fine.** The agent
  may try `add_cart_items_tool` and be refused by a gate. What must not change
  is the cart.

**Run** the next cell. It writes a two-turn probe: a named product with a size
must be added exactly as asked.

In [ ]:
MINE = EVAL / "datasets/val/scripts/probes/P99_notebook_named_size_is_added.yaml"
MINE.write_text('''id: P99_notebook_named_size_is_added
covers: [cart-management, sizes]
why: >
  A shopper who names the product and the size has decided; the add must be
  exactly that, one line, with no follow-up question.
turns:
  - say: show me heels
    expect:
      products_min: 2
  - say: add the Polished Pearl Pumps in size 8
    expect:
      cart:
        - {name: Polished Pearl Pumps, size: "8", qty: 1}
''')
!cd .. && {sys.executable} -m tests.evaluation.src.replay --only P99 --label notebook-mine 2>&1 | tail -5

**You should see** your scenario run and pass:

```
  ok   P99_notebook_named_size_is_added

  pass 1  fail 0  error 0
```

**Try** breaking it on purpose: change the size in `cart:` to `"9"` and run it
again. A check that cannot fail proves nothing.

To keep a scenario, give it the next free number in its folder and a `why`
that says what went wrong when the rule did not hold. Step 11 deletes this one.

## 10. The Challenger and the Judge

Replay confirms what you already know to check. The **Challenger** looks for
what you did not think of: a model plays the shopper from a brief, one turn at
a time, reacting to each reply. Briefs live in
`tests/evaluation/datasets/{text_shopping,image_shopping,style_guide}/`:

```yaml
- id: text_budget_work_bag
  target_turns: 8
  brief: "Shopper wants a work-appropriate bag under a firm budget and will compare options ..."
  constraints:
    - "Firm budget: $60 maximum."
  shopper_behavior:
    type: miserly_strict
```

**Run** the next cell. It is a dry run: it lists what the Challenger would run,
and calls no model.

In [ ]:
!cd .. && PYTHONPATH=tests/evaluation {sys.executable} -m src.challenger --dry-run 2>&1 | head -8

**You should see:**

```
scenario_count: 18
estimated_target_turns: 144
scenarios:
- dataset: text_shopping
  id: text_accessory_for_existing_outfit
...
```

**Do**, to run it live: point the Challenger, and optionally the **Judge**, at
an OpenAI-compatible model. The Judge scores each conversation against
`tests/evaluation/judge_rules.md`, using the cart and product evidence recorded
beside every reply.

```bash
export CHALLENGER_MODEL_BASE_URL="https://integrate.api.nvidia.com/v1"
export CHALLENGER_MODEL_NAME="<model>"
export CHALLENGER_MODEL_API_KEY="$NVIDIA_API_KEY"
PYTHONPATH=tests/evaluation python -m src.challenger --scenario-id text_budget_work_bag

export JUDGE_MODEL_BASE_URL="$CHALLENGER_MODEL_BASE_URL" JUDGE_MODEL_NAME="<model>" JUDGE_MODEL_API_KEY="$NVIDIA_API_KEY"
PYTHONPATH=tests/evaluation python -m src.judge --latest --enable-judge
```

Results open from `tests/evaluation/results/latest.html`. Because the words
change every run, a Challenger failure is a lead, not a verdict. Confirm it by
freezing the turns as a replay scenario (step 9), then fix it until the
scenario passes with repeats (step 7).

## 11. Clean up

**Run** the next cell. It deletes your scenario and this notebook's results.

In [ ]:
import shutil
MINE.unlink(missing_ok=True)
for label in (FIRST, REPEAT, "notebook-mine"):
    shutil.rmtree(RESULTS / label, ignore_errors=True)
clear_shopper()
print("removed", MINE.name, "and the notebook-* results")

**You should see** `removed P99_notebook_named_size_is_added.yaml and the notebook-* results`.

**Where next:**

- [tests/evaluation/datasets/val/README.md](../tests/evaluation/datasets/val/README.md): every scenario and what it proves.
- [tests/evaluation/README.md](../tests/evaluation/README.md): Challenger and Judge configuration.
- [tests/evaluation/PLAN.md](../tests/evaluation/PLAN.md): where evaluation is going.